# 基于deepseek的知识产权合规审查的能力提升

本笔记本在 Google Colab 中运行。运行前请准备：

1. 在 Google Drive 中新建文件夹（默认名 `知识产权`），放入 `标注结果.xlsx`、`law.txt` 以及各平台规则原文 `rule_XX.txt`（编号与标注表一致）。
2. 在 Colab 左侧 🔑（Secrets）中添加 `DEEPSEEK_API_KEY`，并开启本笔记本的访问权限。**不要把 API Key 写进代码。**

## 1. 环境与连接

In [ ]:
!pip install -q openai
from google.colab import drive, userdata
from openai import OpenAI
import pandas as pd, json, re

drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/知识产权/"   # 按你的文件夹名修改
client = OpenAI(api_key=userdata.get("DEEPSEEK_API_KEY"),
                base_url="https://api.deepseek.com")

## 2. 读取标准答案（只取“考卷”平台）

In [ ]:
df = pd.read_excel(BASE + "标注结果.xlsx", sheet_name="标注表", header=3)
df = df[df.iloc[:, 2] == "考卷"]
ITEMS = [c.split("\n")[0] for c in df.columns[3:12]]
print(len(df), "个考卷平台：", list(df.iloc[:, 1]))

## 3. 审查函数
`guide` 为空时即闭卷考；传入开卷资料即开卷考。

In [ ]:
def review(rule_text, guide=""):
    items = "\n".join(ITEMS)
    prompt = f"""你是中国电商法合规审查员。请审查下面这份电商平台知识产权投诉规则，对9个检查项逐项判断，结果只能是：满足、部分满足、缺失、不适用。
{guide}
检查项：
{items}

规则原文：
{rule_text}

只输出JSON，格式为 {{"1": "满足", "2": "部分满足", ...}}，不要输出其他内容。"""
    r = client.chat.completions.create(model="deepseek-chat", temperature=0,
        messages=[{"role": "user", "content": prompt}])
    text = r.choices[0].message.content
    return json.loads(re.search(r"\{.*\}", text, re.S).group())

## 4. 考试与打分

In [ ]:
def run_exam(guide="", name="闭卷"):
    rows = []
    for _, row in df.iterrows():
        no = int(row.iloc[0])
        text = open(f"{BASE}rule_{no:02d}.txt", encoding="utf-8").read()
        pred = review(text, guide)
        for k in range(9):
            gold = row.iloc[3 + k]
            if pd.isna(gold): continue
            p = pred.get(str(k + 1), "")
            rows.append({"平台": row.iloc[1], "检查项": ITEMS[k],
                         "标准答案": gold, "模型判断": p, "正确": p == gold})
        print(row.iloc[1], "完成")
    res = pd.DataFrame(rows)
    res.to_csv(f"{BASE}结果_{name}.csv", index=False, encoding="utf-8-sig")
    print(f"{name}正确率：{res['正确'].mean():.1%}（{res['正确'].sum()}/{len(res)}）")
    return res

## 5. 闭卷考

In [ ]:
closed = run_exam(name="闭卷")
print(closed.groupby("检查项")["正确"].mean().round(2))

## 6. 组装开卷资料

开卷资料 = 法条 + 判断标准（仅通用列）+ 2 道例题（例题平台的原文、标准答案和判断理由）。

**防泄题**：判断标准表中的“疑难案例记录”一列包含考卷平台的判断，不放入开卷资料；考卷平台的原文依据也不放入。

In [ ]:
law = open(BASE + "law.txt", encoding="utf-8").read()

std = pd.read_excel(BASE + "标注结果.xlsx", sheet_name="判断标准", header=3).fillna("")
std_text = ""
for _, s in std.iterrows():
    std_text += (f"{s['编号']}. {s['检查项']}\n法条要求：{s['法条要求']}\n"
                 f"满足：{s['满足']}\n部分满足：{s['部分满足']}\n缺失：{s['缺失']}\n易错点：{s['易错点']}\n\n")
std_text = std_text.replace("（或更短）", "")

all_rows = pd.read_excel(BASE + "标注结果.xlsx", sheet_name="标注表", header=3)
ex_text = ""
for _, e in all_rows[all_rows.iloc[:, 2] == "例题"].iterrows():
    rule = open(f"{BASE}rule_{int(e.iloc[0]):02d}.txt", encoding="utf-8").read()
    ans = {str(k + 1): e.iloc[3 + k] for k in range(9)}
    ex_text += (f"【例题：{e.iloc[1]}】\n规则原文：\n{rule}\n"
                f"标准答案：{json.dumps(ans, ensure_ascii=False)}\n判断理由：{e.iloc[12]}\n\n")

guide = f"""
【相关法条】
{law}

【判断标准】（严格按此标准判断）
{std_text}
【例题】（两份已审查的规则及正确答案，供参考）
{ex_text}
"""
print("开卷资料长度：", len(guide), "字")

## 7. 开卷考并与闭卷对比

In [ ]:
closed = pd.read_csv(BASE + "结果_闭卷.csv")
opened = run_exam(guide=guide, name="开卷")
cmp = pd.DataFrame({"闭卷": closed.groupby("检查项")["正确"].mean(),
                    "开卷": opened.groupby("检查项")["正确"].mean()}).round(2)
print(cmp)

## 8. 错题与逐题变化

In [ ]:
wrong = opened[~opened["正确"]]
print(wrong[["平台", "检查项", "标准答案", "模型判断"]].to_string(), "\n")

m = closed.merge(opened, on=["平台", "检查项"], suffixes=("_闭", "_开"))
up = m[~m["正确_闭"] & m["正确_开"]]
down = m[m["正确_闭"] & ~m["正确_开"]]
print(f"闭卷错→开卷对：{len(up)} 题；闭卷对→开卷错：{len(down)} 题")

## 9. 可复现性检验（开卷再考 2 次）

In [ ]:
for i in (2, 3):
    run_exam(guide=guide, name=f"开卷_{i}")

## 10. 让模型给出引用和理由（用于错题分析）
注意：此处提示词与正式考试不同，结论可能与正式考试不一致；错题以正式考试结果为准。

In [ ]:
def explain(rule_text):
    prompt = f"""你是中国电商法合规审查员。{guide}
检查项：
{chr(10).join(ITEMS)}

规则原文：
{rule_text}

对每个检查项输出：判断结果、引用的规则原文句子、判断理由。"""
    r = client.chat.completions.create(model="deepseek-chat", temperature=0,
        messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content

for no in [4, 6, 7, 12]:   # 按错题所在平台编号调整
    text = open(f"{BASE}rule_{no:02d}.txt", encoding="utf-8").read()
    print(f"===== rule_{no:02d} =====\n", explain(text), "\n")